In [11]:
!pip install gradio==3.50.2 PyMuPDF pandas openpyxl nltk fpdf2 --q

In [12]:
import gradio as gr, fitz, pandas as pd, nltk, random, re, os, io
from fpdf import FPDF; from datetime import datetime
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords; from nltk.tag import pos_tag

[nltk.download(d,quiet=True) for d in ['punkt','stopwords','averaged_perceptron_tagger','punkt_tab','averaged_perceptron_tagger_eng']]

PWD = {"Admin":"adm123","Teacher":"tch123","Student":"stu123","Other":"dem123"}

def extract_text(file):
    try:
        fp=file.name if hasattr(file,'name') else str(file); ext=os.path.splitext(fp)[-1].lower()
        if ext==".pdf":
            d=fitz.open(fp); t=" ".join(p.get_text() for p in d); d.close(); return t
        return " ".join((pd.read_excel(fp) if ext in(".xlsx",".xls") else pd.read_csv(fp)).astype(str).values.flatten())
    except: return ""

def get_sents(text):
    sents=[]
    try: sents=sent_tokenize(text)
    except: pass
    if len(sents)<3:
        parts=re.split(r'[\n\r;:|]+',text); sents=[p.strip() for p in parts if len(p.strip())>5]
    if len(sents)<3:
        sents=re.split(r'[.!?,]+',text)
    return [x.strip() for x in sents if len(x.split())>2]

def get_kws(text,n=50):
    stop=set(stopwords.words('english'))
    try: w=[w for w,t in pos_tag(word_tokenize(text.lower())) if t in('NN','NNS','NNP','NNPS','JJ') and w.isalpha() and w not in stop and len(w)>3]
    except: w=[x for x in text.lower().split() if len(x)>4]
    f={}
    for x in w: f[x]=f.get(x,0)+1
    return sorted(f,key=f.get,reverse=True)[:n]

def gen_mcqs(sents,kws,n=10):
    stop=set(stopwords.words('english'))
    out=[]; s_copy=sents[:]*3; random.shuffle(s_copy)
    for s in s_copy:
        if len(out)>=n: break
        words=s.split()
        if len(words)<3: continue
        try: nouns=[w for w,t in pos_tag(word_tokenize(s)) if t in('NN','NNS','NNP') and w.isalpha() and len(w)>3]
        except: nouns=[]
        if not nouns: nouns=[w for w in words if w.isalpha() and len(w)>3 and w.lower() not in stop]
        if not nouns: continue
        a=random.choice(nouns)
        opts=[k for k in kws if k.lower()!=a.lower() and k.isalpha()][:3]
        if len(opts)<3:
            extras=[w for w in words if w.isalpha() and w.lower()!=a.lower() and len(w)>2]
            opts=(opts+extras)[:3]
        if len(opts)<3: continue
        opts=opts+[a]; random.shuffle(opts)
        q_text=f"Fill in the blank: {s.replace(a,'_______',1)}"
        if any(x['q']==q_text for x in out): continue
        out.append({"q":q_text,"o":opts,"a":chr(65+opts.index(a)),"aw":a})
    return out

def gen_short(sents,n=8):
    out=[]; used=set(); s_copy=sents[:]; random.shuffle(s_copy)
    for i,s in enumerate(s_copy):
        if len(out)>=n or s in used: continue
        try: nouns=[w for w,t in pos_tag(word_tokenize(s)) if t in('NN','NNS','NNP','NNPS') and len(w)>3]
        except: nouns=[]
        topic=nouns[0] if nouns else s.split()[0]
        rel=[x for x in sents if topic.lower() in x.lower() and len(x.split())>2][:8] or sents[max(0,i-3):i+4]
        ans=" ".join(list(dict.fromkeys([x.strip() for x in rel if len(x.strip())>10])))
        out.append({"q":f"Explain in brief: what is '{topic}' and its significance?","a":ans or s}); used.add(s)
    return out

def gen_long(sents,n=5):
    E,M,H=[],[],[]; entries=[]
    for i,s in enumerate(sents):
        if len(s.split())<3: continue
        sc=len(s.split())+len([w for w in s.split() if len(w)>8])*2
        d="E" if sc<20 else("M" if sc<40 else "H")
        rel=" ".join(list(dict.fromkeys([x.strip() for x in sents[max(0,i-4):i+6] if len(x.strip())>10]))[:10])
        e={"q":f"Explain in detail: {s[:80]}{'...'if len(s)>80 else ''}","a":rel or s,"d":d}
        entries.append(e)
        if d=="E" and len(E)<n: E.append(e)
        elif d=="M" and len(M)<n: M.append(e)
        elif d=="H" and len(H)<n: H.append(e)
    random.shuffle(entries)
    for e in entries:
        if len(E)>=n and len(M)>=n and len(H)>=n: break
        if len(E)<n: E.append(e)
        elif len(M)<n: M.append(e)
        elif len(H)<n: H.append(e)
    return E[:n],M[:n],H[:n]

def format_paper(title,subject,mcqs,shorts,E,M,H):
    now=datetime.now().strftime("%d %B %Y | %H:%M")
    total=len(mcqs)*2+len(shorts)*5+(len(E)+len(M)+len(H))*10
    L=["="*72,"         AI GENERATED QUESTION PAPER","="*72,
       f"  Title      : {title}",f"  Subject    : {subject}",
       f"  Date       : {now}",f"  Total Marks: {total}","="*72,
       f"\n{'-'*72}  SECTION A - MCQs  [{len(mcqs)*2} Marks]\n{'-'*72}"]
    for i,m in enumerate(mcqs,1):
        L.append(f"\nQ{i}. {m['q']}")
        for j,o in enumerate(m['o']): L.append(f"     {chr(65+j)}) {o}")
        L.append(f"     Answer: ({m['a']}) {m['aw']}")
    L+=[f"\n{'-'*72}  SECTION B - SHORT ANSWERS  [{len(shorts)*5} Marks]\n{'-'*72}"]
    for i,s in enumerate(shorts,1): L.append(f"\nQ{i}. {s['q']}\n     Model Answer: {s['a'][:300]}")
    L+=[f"\n{'-'*72}  SECTION C - LONG ANSWERS  [10 Marks each]\n{'-'*72}"]
    for lbl,lst in [("EASY LEVEL",E),("MEDIUM LEVEL",M),("HARD LEVEL",H)]:
        L.append(f"\n  >> {lbl}\n  "+"-"*60)
        for i,q in enumerate(lst,1): L.append(f"\n  {lbl[0]}{i}. {q['q']}\n      Model Answer: {q['a'][:400]}")
    L+=["\n"+"="*72,"              *** END OF QUESTION PAPER ***","="*72]
    return "\n".join(L)

# ── DEFINITIVE PDF FIX: pass path directly to pdf.output(path) ──
def export_pdf(content, subject="subject"):
    try:
        path = f"/content/{re.sub(r'[^a-zA-Z0-9_]','_',subject)}_question_paper.pdf"
        pdf = FPDF()
        pdf.set_auto_page_break(auto=True, margin=15)
        pdf.add_page()
        pdf.set_text_color(0, 0, 0)
        for line in content.split('\n'):
            safe = line.encode('latin-1','replace').decode('latin-1')
            if safe.strip() == '':
                pdf.ln(3); continue
            hdr = any(x in safe for x in ['AI GENERATED','*** END','===','SECTION'])
            isq = safe.strip().startswith('Q') and len(safe.strip())>2 and safe.strip()[1:2].isdigit()
            if hdr or isq: pdf.set_font("Courier",'B',10)
            elif 'Answer:' in safe: pdf.set_font("Courier",'I',9)
            else: pdf.set_font("Courier",'',9)
            pdf.multi_cell(0, 5, safe)
        pdf.output(path)   # ← writes directly to file, works in ALL fpdf2 versions
        print(f"PDF OK: {path} | size: {os.path.getsize(path)} bytes")
        return path
    except Exception as e:
        import traceback; traceback.print_exc()
        return None

CHAT=[
    (["hello","hi","hey"],"Hello! I am your AI Question Paper Assistant. Ask me about MCQs, answers, PDF download, passwords, or how to use the app!"),
    (["how","help","use","start","guide","steps"],"How to use:\n1. Paste your syllabus text in the text box\n2. Or upload a PDF/Excel/CSV\n3. Enter Exam Title and Subject Name\n4. Set sliders for number of questions\n5. Click 'Generate Question Paper'\n6. Download PDF from the link below preview"),
    (["mcq","multiple","fill","objective"],"MCQs are fill-in-the-blank questions.\nKey word is blanked out from syllabus.\n4 options (A,B,C,D) given.\nCorrect answer shown after each question.\nEach MCQ = 2 marks."),
    (["short","brief","5 mark","section b"],"Short Answers ask about a key concept from syllabus.\nModel answer (up to 8 sentences) provided for teachers.\nEach short answer = 5 marks."),
    (["long","essay","detail","10 mark","section c","elaborate"],"Long Answers are divided into 3 difficulty levels:\n• Easy Level\n• Medium Level\n• Hard Level\nDetailed model answer included.\nEach long answer = 10 marks."),
    (["difficult","level","easy","medium","hard"],"Difficulty is auto-detected:\n• Easy: short sentences\n• Medium: moderate length\n• Hard: long technical sentences"),
    (["password","login","role","admin","teacher","student","credential"],"Default passwords:\n• Admin   → adm123\n• Teacher → tch123\n• Student → stu123\n• Other   → dem123"),
    (["pdf","download","save","export","print"],"After generating, a PDF file link appears below the preview.\nClick it to download.\nSaved in /content/ folder in Colab."),
    (["syllabus","paste","upload","text","content","input"],"You can:\n• Paste text directly into the paste box\n• Upload PDF/Excel/CSV\nBoth work independently. More content = better questions."),
    (["marks","total","score","grading"],"Marks:\n• MCQs = 2 marks each\n• Short answers = 5 marks each\n• Long answers = 10 marks each\nTotal shown in paper header."),
    (["slider","number","how many","count","adjust","questions"],"Sliders in Settings control:\n• MCQs: 1-20\n• Short answers: 1-15\n• Long answers per level: 1-10"),
    (["answer","key","solution","correct","model"],"All answers included:\n• MCQs: correct letter + answer word\n• Short: full model answer\n• Long: detailed model answer"),
    (["error","not enough","problem","fail","wrong","sentence"],"If error appears:\n• Paste at least a few lines of text\n• More content = more questions\n• Try 2-3 paragraphs\n• Reduce slider values if syllabus is short"),
]
def chatbot_respond(msg,history):
    u=msg.lower().strip()
    r=next((r for k,r in CHAT if any(x in u for x in k)),
           "I can help with:\n• How to use the app\n• MCQ/Short/Long answer questions\n• PDF download\n• Login passwords\n• Marks and scoring\n• Slider settings")
    return (history or[])+[(msg,r)],""

def login(name,role,pw):
    if not name.strip(): return gr.update(visible=True),gr.update(visible=False),"Please enter your name."
    if not role: return gr.update(visible=True),gr.update(visible=False),"Please select your role."
    if pw==PWD.get(role,"dem123"): return gr.update(visible=False),gr.update(visible=True),f"Welcome {name}! Logged in as {role}."
    return gr.update(visible=True),gr.update(visible=False),"Wrong password. Try again."
def logout(): return gr.update(visible=True),gr.update(visible=False),"Logged out. Please login to continue."

def generate_paper(file,pasted,title,subject,nm,ns,nd):
    text=((extract_text(file) if file else "")+" "+(str(pasted).strip() if pasted else "")).strip()
    if len(text)<30: return f"Please paste syllabus text. Got only {len(text)} chars.",None
    sents=get_sents(text)
    if len(sents)<2: return f"Could not split content ({len(sents)} parts). Try pasting with newlines or punctuation.",None
    kws=get_kws(text) or [w for w in text.lower().split() if len(w)>4][:50]
    E,M,H=gen_long(sents,int(nd))
    result=format_paper(title,subject,gen_mcqs(sents,kws,int(nm)),gen_short(sents,int(ns)),E,M,H)
    pdf_path=export_pdf(result,subject)
    return result, pdf_path

_sd=[(3,6,4,0),(10,8,6,.5),(18,5,5,1),(26,9,7,1.5),(34,6,4.5,2),(42,7,6.5,.8),(51,8,5.5,2.5),(59,5,4,3),(67,10,7,.3),(75,7,5,1.8),(83,5,6,2.2),(91,8,4.8,.6),(97,6,5.5,3.5),(22,5,7.5,4),(48,7,4.2,1.2)]
_qd=[(6,7,0),(16,9,1),(28,8,2),(39,10,.5),(52,7.5,3),(63,8.5,1.5),(74,9.5,2.5),(85,7,4),(93,10,3.5),(11,8,.8)]
_sc="".join(f".sparkle:nth-child({i+1}){{left:{d[0]}%;width:{d[1]}px;height:{d[1]}px;animation-duration:{d[2]}s;animation-delay:{d[3]}s}}" for i,d in enumerate(_sd))
_qc="".join(f".qpaper:nth-child({i+16}){{left:{d[0]}%;animation-duration:{d[1]}s;animation-delay:{d[2]}s}}" for i,d in enumerate(_qd))
css="""
body{background:radial-gradient(ellipse at top,#000428,#004e92,#000428);background-size:400% 400%;animation:bg 12s ease infinite;font-family:'Segoe UI',sans-serif;overflow-x:hidden}
@keyframes bg{0%{background-position:0% 50%}50%{background-position:100% 50%}100%{background-position:0% 50%}}
.sparkle{position:fixed;border-radius:50%;background:radial-gradient(circle,#00ffff,#0055ff);box-shadow:0 0 12px 5px rgba(0,230,255,.95);pointer-events:none;z-index:0;animation:sparklefall linear infinite;opacity:0}
@keyframes sparklefall{0%{top:-20px;opacity:0;transform:scale(.4) rotate(0deg)}10%{opacity:1}50%{opacity:.9;transform:scale(1.4) rotate(180deg)}90%{opacity:.6}100%{top:110vh;opacity:0;transform:scale(.2) rotate(360deg)}}
"""+_sc+"""
.qpaper{position:fixed;width:38px;height:50px;background:rgba(255,255,255,.08);border:2px solid rgba(0,220,255,.7);border-radius:4px;pointer-events:none;z-index:0;animation:paperfall linear infinite;opacity:0;box-shadow:0 0 10px rgba(0,200,255,.5)}
.qpaper::before{content:'';display:block;margin:8px 6px 0;height:2px;background:rgba(0,220,255,.7);box-shadow:0 6px 0 rgba(0,220,255,.5),0 12px 0 rgba(0,220,255,.4),0 18px 0 rgba(0,220,255,.3)}
@keyframes paperfall{0%{top:-70px;opacity:0;transform:rotate(-10deg) scale(.8)}8%{opacity:.9}60%{opacity:.7}100%{top:110vh;opacity:0;transform:rotate(12deg) scale(.7)}}
"""+_qc+"""
#login_box{background:linear-gradient(135deg,rgba(0,60,120,.75),rgba(0,20,80,.85));border:2px solid rgba(0,220,255,.9);border-radius:24px;padding:40px 35px;max-width:460px;margin:50px auto;box-shadow:0 0 70px rgba(0,220,255,.6),0 0 130px rgba(0,120,255,.3);backdrop-filter:blur(18px);animation:loginpulse 3s ease-in-out infinite;position:relative;z-index:1}
@keyframes loginpulse{0%,100%{box-shadow:0 0 70px rgba(0,220,255,.5),0 0 130px rgba(0,120,255,.2)}50%{box-shadow:0 0 100px rgba(0,255,255,.8),0 0 180px rgba(0,180,255,.4)}}
#dashboard_box{background:rgba(0,10,40,.5);border:1px solid rgba(0,191,255,.3);border-radius:20px;padding:25px;backdrop-filter:blur(10px);position:relative;z-index:1}
label{color:#a0d8ff!important;font-weight:700!important}
input,textarea,select{background:rgba(0,20,60,.9)!important;border:2px solid rgba(0,150,255,.5)!important;color:#fff!important;border-radius:10px!important}
#output_paper textarea{min-height:500px!important;font-family:'Courier New',monospace!important;font-size:.88rem!important;line-height:1.7!important;color:#000!important;background:#fff!important;border:2px solid #aaa!important;border-radius:10px!important;padding:20px!important}
.gr-button-primary{background:linear-gradient(135deg,#00c6ff,#0072ff)!important;border:none!important;color:#fff!important;font-weight:800!important;border-radius:12px!important}
.gr-button-secondary{background:linear-gradient(135deg,#1a1a4e,#0d3b8e)!important;border:1px solid rgba(0,191,255,.5)!important;color:#00cfff!important;border-radius:12px!important}
input[type=range]{accent-color:#00cfff!important}
"""

with gr.Blocks(css=css,title="AI Question Paper Generator") as app:
    gr.HTML("".join(['<div class="sparkle"></div>']*15)+"".join(['<div class="qpaper"></div>']*10))
    gr.HTML("""<div style='text-align:center;padding:28px 20px 12px;position:relative;z-index:2;'>
        <div style='display:inline-block;background:rgba(0,5,30,.85);border:2px solid rgba(0,255,255,.6);border-radius:18px;padding:18px 40px 14px;box-shadow:0 0 40px rgba(0,200,255,.5),0 0 80px rgba(0,100,255,.3);backdrop-filter:blur(10px);'>
        <h1 style='color:#00ffff;font-size:2.5rem;font-weight:900;margin:0 0 6px;letter-spacing:3px;text-shadow:0 0 15px #00ffff,0 0 35px #00cfff,0 0 70px #0088ff;'>&#127891; AI QUESTION PAPER GENERATOR</h1>
        <p style='color:#e0f8ff;font-size:1rem;margin:0;font-weight:600;'>Upload or Paste Syllabus &rarr; Auto-generate MCQs, Short and Long Answers with Answer Key</p>
        </div></div>""")
    status_box=gr.Textbox(value="Please login to continue...",label="Status",interactive=False)
    with gr.Column(visible=True,elem_id="login_box") as lp:
        gr.HTML("""<div style='text-align:center;margin-bottom:24px;'>
            <h2 style='color:#00ffff;font-size:1.9rem;font-weight:900;margin:0 0 6px;letter-spacing:2px;text-shadow:0 0 20px #00ffff,0 0 40px #00cfff;'>&#128272; LOGIN PORTAL</h2>
            <p style='color:#fff;font-size:.95rem;margin:0;font-weight:600;'>Enter your details to access the dashboard</p></div>""")
        ni=gr.Textbox(label="Full Name",placeholder="e.g. Priya Sharma")
        ri=gr.Dropdown(label="Role",choices=["Admin","Teacher","Student","Other"])
        pi=gr.Textbox(label="Password",type="password")
        gr.HTML("""<div style='color:#fff;font-size:.83rem;padding:10px 14px;border:1px solid rgba(0,220,255,.5);border-radius:8px;background:rgba(0,80,160,.3);font-weight:600;margin-top:6px;'>
            <b style='color:#00ffff;'>Default Passwords:</b><br>Admin &rarr; adm123 &nbsp;|&nbsp; Teacher &rarr; tch123 &nbsp;|&nbsp; Student &rarr; stu123 &nbsp;|&nbsp; Other &rarr; dem123</div>""")
        lb=gr.Button("Login ->",variant="primary")
    with gr.Column(visible=False,elem_id="dashboard_box") as dp:
        gr.HTML("<h2 style='color:#00cfff;text-align:center;font-weight:800;'>Dashboard</h2>")
        with gr.Row():
            with gr.Column(scale=1,min_width=300):
                gr.HTML("<h3 style='color:#00cfff;'>Syllabus Input</h3>")
                fi=gr.File(label="Upload PDF / Excel / CSV",file_types=[".pdf",".xlsx",".xls",".csv"])
                ti=gr.Textbox(label="Paste Syllabus Text Here",placeholder="Copy and paste your syllabus content here...",lines=10)
                gr.HTML("<div style='color:#80ffb0;font-size:.8rem;padding:6px 10px;background:rgba(0,100,50,.2);border-radius:8px;border:1px solid rgba(0,255,150,.3);'>You can use ONLY the paste box without uploading any file.</div>")
                tli=gr.Textbox(label="Exam Title",value="Annual Examination 2025-26")
                si=gr.Textbox(label="Subject Name",value="Computer Science")
                gr.HTML("<h3 style='color:#00cfff;margin-top:15px;'>Settings</h3>")
                mn=gr.Slider(1,20,value=10,step=1,label="MCQs")
                sn=gr.Slider(1,15,value=8,step=1,label="Short Answer Questions")
                dn=gr.Slider(1,10,value=5,step=1,label="Long Answer Questions per Level")
                with gr.Row():
                    gb=gr.Button("Generate Question Paper",variant="primary")
                    ob=gr.Button("Logout",variant="secondary")
                gr.HTML("<h3 style='color:#00cfff;margin-top:20px;'>AI Assistant</h3>")
                cb=gr.Chatbot(height=280)
                ci=gr.Textbox(placeholder="Ask me anything about the app...",label="Message")
                csb=gr.Button("Send",variant="secondary")
            with gr.Column(scale=2):
                gr.HTML("<h3 style='color:#00cfff;'>Question Paper Preview</h3>")
                ot=gr.Textbox(lines=35,max_lines=100,interactive=False,elem_id="output_paper",label="Generated Paper")
                gr.HTML("<div style='color:#80ffb0;font-size:.85rem;padding:8px 12px;background:rgba(0,80,40,.3);border-radius:8px;border:1px solid rgba(0,255,120,.4);margin:6px 0;'>Click the filename below to download your PDF.</div>")
                of=gr.File(label="Download PDF")

    lb.click(fn=login,inputs=[ni,ri,pi],outputs=[lp,dp,status_box])
    ob.click(fn=logout,inputs=[],outputs=[lp,dp,status_box])
    gb.click(fn=generate_paper,inputs=[fi,ti,tli,si,mn,sn,dn],outputs=[ot,of])
    csb.click(fn=chatbot_respond,inputs=[ci,cb],outputs=[cb,ci])
    ci.submit(fn=chatbot_respond,inputs=[ci,cb],outputs=[cb,ci])

app.launch(share=True,debug=True)

IMPORTANT: You are using gradio version 3.50.2, however version 4.44.1 is available, please upgrade.
--------
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Running on public URL: https://1fec5d9577ebc06f77.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


PDF OK: /content/Computer_Science_question_paper.pdf | size: 4465 bytes
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://1fec5d9577ebc06f77.gradio.live
